In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# 加载文件数据
def load_house_data():
    data = np.loadtxt("./data/house.txt", delimiter=",", skiprows=1)
    X = data[:, :4]
    Y = data[:, 4]
    return X, Y

In [4]:
X_train, Y_train = load_house_data()
# m,n分别对应行数和列数,m为房子套数,n为特征数量
m, n = X_train.shape

In [ ]:
"""代价函数"""


def compute_cost(X, Y, w, b):
    m = X.shape[0]
    f_wb = X @ w + b  # @-> m*n @ 1*n,点积
    cost = np.sum((f_wb - Y) ** 2) / (2 * m)
    return cost

In [6]:
"""梯度计算"""


def compute_gradient(X, Y, w, b):
    m = X.shape[0]
    f_wb = X @ w + b
    dj_dw = (X.T @ (f_wb - Y)) / m
    dj_db = np.sum(f_wb - Y) / m
    return dj_dw, dj_db

In [7]:
"""梯度下降"""


def gradient_descent(X, Y, w_init, b_init, alpha, num_iters):
    w = w_init.copy()
    b = b_init
    cost_history = []

    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, Y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        cost_history.append(compute_cost(X, Y, w, b))

        # 每10次打印一次进度
        if i % max(1, num_iters // 10) == 0:
            print(f"Iter {i:4d}: Cost = {cost_history[-1]:.4e}")

        return w, b, cost_history

In [8]:
"""特征缩放"""


def zscore_normalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    X_norm = (X - mu) / sigma
    return X_norm, mu, sigma

In [ ]:
# 原始数据（未缩放）
print("===== 未缩放特征 =====")
w_init = np.zeros(n)
b_init = 0.0
alpha = 1e-7  # 未缩放时学习率必须很小
iterations = 500
w_final, b_final, cost_hist = gradient_descent(
    X_train, Y_train, w_init, b_init, alpha, iterations
)
print(f"最终 w = {w_final}, b = {b_final:.2f}\n")

# 特征缩放
X_norm, mu, sigma = zscore_normalize(X_train)

# 缩放后重新运行（可以用更大的学习率）
print("===== 缩放后特征（Z-score） =====")
w_init = np.zeros(n)
b_init = 0.0
alpha = 0.1  # 缩放后学习率可以大很多
iterations = 500
w_final_norm, b_final_norm, cost_hist_norm = gradient_descent(
    X_norm, Y_train, w_init, b_init, alpha, iterations
)
print(f"最终 w = {w_final_norm}, b = {b_final_norm:.2f}")

# ---------------------------- 对比学习率的影响（仅演示） ----------------------------
print("\n===== 不同学习率对比（缩放后） =====")
alphas = [0.01, 0.1, 0.8]
for a in alphas:
    w, b, _ = gradient_descent(X_norm, Y_train, np.zeros(n), 0.0, a, 100)
    print(f"α = {a}: 最终代价 = {compute_cost(X_norm, Y_train, w, b):.4e}")
